In [1]:
# Importing the Libraries
import numpy as np # linear algebra
import pandas as pd # data processing
import os
import string
from string import digits
import matplotlib.pyplot as plt
%matplotlib inline
import re
import seaborn as sns
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from keras.layers import Input, LSTM, Embedding, Dense
from keras.models import Model

In [2]:
# Loading the data
lines = pd.read_csv('Hindi_English_Truncated_Corpus.csv', encoding = 'utf-8')
lines = lines[lines['source'] == 'ted']
lines = lines[~pd.isnull(lines['english_sentence'])]
lines.drop_duplicates(inplace= True)
lines = lines.sample(n = 25000, random_state= 42)
lines.shape


(25000, 3)

In [3]:
# For simplicity let's lowercase all the characters in the dataset
lines['english_sentence'] = lines['english_sentence'].apply(lambda x: x.lower())
lines['hindi_sentence'] = lines['hindi_sentence'].apply(lambda x: x.lower())

In [4]:
# Now let's remove all the quotes from the data
lines['english_sentence']=lines['english_sentence'].apply(lambda x: re.sub("'", '', x))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: re.sub("'", '', x))

In [5]:
# Now let's remove all the special characters in the data
exclude = set(string.punctuation) # Set of all special characters
# Remove all the special characters
lines['english_sentence']=lines['english_sentence'].apply(lambda x: ''.join(ch for ch in x if ch not in exclude))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: ''.join(ch for ch in x if ch not in exclude))



In [6]:
# Now let's remove all the numbers and extra spaces from the data
remove_digits = str.maketrans('', '', digits)
lines['english_sentence']=lines['english_sentence'].apply(lambda x: x.translate(remove_digits))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: x.translate(remove_digits))
lines['hindi_sentence'] = lines['hindi_sentence'].apply(lambda x: re.sub("[२३०८१५७९४६]", "", x))

In [7]:

# Remove extra spaces
lines['english_sentence']=lines['english_sentence'].apply(lambda x: x.strip())
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: x.strip())
lines['english_sentence']=lines['english_sentence'].apply(lambda x: re.sub(" +", " ", x))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: re.sub(" +", " ", x))
lines['hindi_sentence'] = lines['hindi_sentence'].apply(lambda x : 'START_ '+ x + ' _END')

In [8]:
### Get English and Hindi Vocabulary
all_eng_words=set()
for eng in lines['english_sentence']:
    for word in eng.split():
        if word not in all_eng_words:
            all_eng_words.add(word)
    all_hindi_words=set()
    for hin in lines['hindi_sentence']:
        for word in hin.split():
            if word not in all_hindi_words:
                all_hindi_words.add(word)
lines['length_eng_sentence']=lines['english_sentence'].apply(lambda x:len(x.split(" ")))
lines['length_hin_sentence']=lines['hindi_sentence'].apply(lambda x:len(x.split(" ")))

In [9]:
# Now before training the language translation model we need to set the input and target values
lines=lines[lines['length_eng_sentence']<=20]
lines=lines[lines['length_hin_sentence']<=20]
max_length_src=max(lines['length_hin_sentence'])
max_length_tar=max(lines['length_eng_sentence'])
input_words = sorted(list(all_eng_words))
target_words = sorted(list(all_hindi_words))
num_encoder_tokens = len(all_eng_words)
num_decoder_tokens = len(all_hindi_words)
num_encoder_tokens, num_decoder_tokens
num_decoder_tokens += 1 #for zero padding
input_token_index = dict([(word, i+1) for i, word in enumerate(input_words)])
target_token_index = dict([(word, i+1) for i, word in enumerate(target_words)])
reverse_input_char_index = dict((i, word) for word, i in input_token_index.items())
reverse_target_char_index = dict((i, word) for word, i in target_token_index.items())
lines = shuffle(lines)

Training Model to Translate English to Hindi


In [10]:
X, y = lines['english_sentence'], lines['hindi_sentence']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2,random_state=42)
X_train.to_pickle('X_train.pkl')
X_test.to_pickle('X_test.pkl')

In [11]:
# Now let’s train our language translation model
def generate_batch(X = X_train, y = y_train, batch_size = 128):
    ''' Generate a batch of data '''
    while True:
        for j in range(0, len(X), batch_size):
                encoder_input_data = np.zeros((batch_size, max_length_src),dtype='float32')
                decoder_input_data = np.zeros((batch_size, max_length_tar),dtype='float32')
                decoder_target_data = np.zeros((batch_size, max_length_tar, num_decoder_tokens),dtype='float32')
                for i, (input_text, target_text) in enumerate(zip(X[j:j+batch_size], y[j:j+batch_size])):
                    for t, word in enumerate(input_text.split()):
                        encoder_input_data[i, t] = input_token_index[word] # encoder input seq
                        for t, word in enumerate(target_text.split()):
                            if t<len(target_text.split())-1:
                                decoder_input_data[i, t] = target_token_index[word] # decoder input seq
                                if t>0:
# decoder target sequence (one hot encoded)
# does not include the START_ token
# Offset by one timestep
                                    decoder_target_data[i, t - 1, target_token_index[word]] = 1.
                                    yield([encoder_input_data, decoder_input_data], decoder_target_data)

In [12]:
latent_dim=300
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(num_encoder_tokens, latent_dim, mask_zero = True)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
# We discard `encoder_outputs` and only keep the states.
encoder_states = [state_h, state_c]
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(num_decoder_tokens, latent_dim, mask_zero = True)
dec_emb = dec_emb_layer(decoder_inputs)
# We set up our decoder to return full output sequences,
# and to return internal states as well. We don't use the
# return states in the training model, but we will use them in inference.
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb,
initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [17]:
def generate_batch(X, y, batch_size=128):
    while True:
        for j in range(0, len(X), batch_size):
            encoder_input_data = np.array(X[j:j+batch_size])   # (batch, input_len)
            decoder_input_data = np.array(y[j:j+batch_size, :-1])  # shifted input
            decoder_target_data = np.array(y[j:j+batch_size, 1:])  # shifted target

            # one-hot encode targets
            decoder_target_data = tf.keras.utils.to_categorical(
                decoder_target_data,
                num_classes=num_decoder_tokens
            )

            yield ([encoder_input_data, decoder_input_data], decoder_target_data)


In [20]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Fit tokenizer on target language (Hindi sentences)
y_tokenizer = Tokenizer(filters='', lower=False, split=' ')
y_tokenizer.fit_on_texts(y_train)

# Convert sentences to integer sequences
y_train_seq = y_tokenizer.texts_to_sequences(y_train)
y_test_seq = y_tokenizer.texts_to_sequences(y_test)

# Vocabulary size
num_decoder_tokens = len(y_tokenizer.word_index) + 1
print("Decoder vocab size:", num_decoder_tokens)


Decoder vocab size: 15354


In [24]:
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)


In [22]:
max_decoder_seq_length = max(len(seq) for seq in y_train_seq)

y_train = pad_sequences(y_train_seq, maxlen=max_decoder_seq_length, padding='post')
y_test = pad_sequences(y_test_seq, maxlen=max_decoder_seq_length, padding='post')


In [23]:
x_tokenizer = Tokenizer(filters='', lower=False, split=' ')
x_tokenizer.fit_on_texts(X_train)

X_train_seq = x_tokenizer.texts_to_sequences(X_train)
X_test_seq = x_tokenizer.texts_to_sequences(X_test)

max_encoder_seq_length = max(len(seq) for seq in X_train_seq)

X_train = pad_sequences(X_train_seq, maxlen=max_encoder_seq_length, padding='post')
X_test = pad_sequences(X_test_seq, maxlen=max_encoder_seq_length, padding='post')

num_encoder_tokens = len(x_tokenizer.word_index) + 1
print("Encoder vocab size:", num_encoder_tokens)


Encoder vocab size: 12443


In [26]:
def generate_batch(X, y, batch_size=128):
    while True:
        for j in range(0, len(X), batch_size):
            encoder_input_data = np.array(X[j:j+batch_size])   # (batch, input_len)
            decoder_input_data = np.array(y[j:j+batch_size, :-1])  # shifted input
            decoder_target_data = np.array(y[j:j+batch_size, 1:])  # shifted target

            # one-hot encode targets
            decoder_target_data = tf.keras.utils.to_categorical(
                decoder_target_data,
                num_classes=num_decoder_tokens
            )

            yield ([encoder_input_data, decoder_input_data], decoder_target_data)


In [27]:
history = model.fit(
    generate_batch(X_train, y_train, batch_size=batch_size),
    steps_per_epoch=train_samples // batch_size,
    epochs=epochs,
    validation_data=generate_batch(X_test, y_test, batch_size=batch_size),
    validation_steps=val_samples // batch_size
)

model.save_weights('nmt_weights.h5')


NameError: name 'tf' is not defined